#**TASK 2: Data Engineering**

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import OneHotEncoder
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StandardScaler
from pyspark.ml import Pipeline
from pyspark.sql.functions import col
from pyspark.sql.functions import *
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import OneHotEncoder
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StandardScaler
from pyspark.ml import Pipeline


#Creation of Spark Session

In [2]:

spark = SparkSession.builder \
    .appName("Task2_Data_Engineering") \
    .getOrCreate()

#Loading the Dataset

In [3]:

df = spark.read.csv(
    "/content/DelayData.csv",
    header=True,
    inferSchema=True
)

#Partition Count

In [4]:
(df.rdd.getNumPartitions())


2

#Missing Value Check

In [5]:

for column in df.columns:
    missing = df.filter(col(column).isNull()).count()
    print(f"{column}: {missing}")


depdelay: 0
arrdelay: 0
scheduleddepartdatetime: 0
origin: 0
dest: 0
uniquecarrier: 0
marketshareorigin: 0
marketsharedest: 0
hhiorigin: 0
hhidest: 0
nonhubairportorigin: 0
smallhubairportorigin: 0
mediumhubairportorigin: 0
largehubairportorigin: 0
nonhubairportdest: 0
smallhubairportdest: 0
mediumhubairportdest: 0
largehubairportdest: 0
nonhubairlineorigin: 0
smallhubairlineorigin: 0
mediumhubairlineorigin: 0
largehubairlineorigin: 0
nonhubairlinedest: 0
smallhubairlinedest: 0
mediumhubairlinedest: 0
largehubairlinedest: 0
year: 0
month: 0
dayofmonth: 0
dayofweek: 0
scheduledhour: 0
originairportid: 0
destairportid: 0
tailnum: 0
capacity: 0
loadfactor: 0
numflights: 0
origincityname: 0
originstate: 0
distance: 0
monopolyroute: 0
temperature: 0
temp_ninfty_n10: 0
temp_n10_0: 0
temp_0_10: 0
temp_10_20: 0
temp_20_30: 0
temp_30_40: 0
temp_40_infty: 0
windspeed: 0
windspeedsquare: 0
windgustdummy: 0
windgustspeed: 0
raindummy: 0
raintracedummy: 0
snowdummy: 0
snowtracedummy: 0
originmetrop

# Feature Engineering

In [6]:


# Target Variable

df = df.withColumn(
    "delay_label",
    when(col("arrdelay") > 15, 1).otherwise(0)
)

# Usage of the existing scheduled hour

df = df.withColumn(
    "departure_hour",
    col("scheduledhour")
)

# Distance Category

df = df.withColumn(
    "distance_category",
    when(col("distance") < 500, "Short Haul")
    .when((col("distance") >= 500) & (col("distance") < 1500), "Medium Haul")
    .otherwise("Long Haul")
)

# Checking the Results

df.select(
    "arrdelay",
    "delay_label",
    "scheduledhour",
    "departure_hour",
    "distance",
    "distance_category"
).show(10, truncate=False)

+--------+-----------+-------------+--------------+--------+-----------------+
|arrdelay|delay_label|scheduledhour|departure_hour|distance|distance_category|
+--------+-----------+-------------+--------------+--------+-----------------+
|-4.0    |0          |15           |15            |496     |Short Haul       |
|11.0    |0          |14           |14            |425     |Short Haul       |
|12.0    |0          |12           |12            |1391    |Medium Haul      |
|24.0    |1          |15           |15            |2401    |Long Haul        |
|-8.0    |0          |18           |18            |422     |Short Haul       |
|-2.0    |0          |20           |20            |432     |Short Haul       |
|-4.0    |0          |12           |12            |528     |Medium Haul      |
|5.0     |0          |9            |9             |229     |Short Haul       |
|2.0     |0          |14           |14            |337     |Short Haul       |
|1.0     |0          |21           |21            |6

#Feature Selection

In [7]:

# Feature Selection

selected_features = [
    "distance",
    "temperature",
    "windspeed",
    "capacity",
    "loadfactor",
    "scheduledhour",
    "delay_label"
]

feature_selected_df = df.select(selected_features)

# Showing the selected features

feature_selected_df.show(10, truncate=False)

# Displaying the schema

feature_selected_df.printSchema()

# Displaying dimensions

print("Rows:", feature_selected_df.count())
print("Columns:", len(feature_selected_df.columns))

+--------+-----------------+----------------+--------+----------+-------------+-----------+
|distance|temperature      |windspeed       |capacity|loadfactor|scheduledhour|delay_label|
+--------+-----------------+----------------+--------+----------+-------------+-----------+
|496     |15.34            |10.3            |2       |0.46060732|15           |0          |
|425     |11.5             |13.6666666666667|2       |0.61790389|14           |0          |
|1391    |12.2188679245283 |13.5283018867925|142     |0.45592913|12           |0          |
|2401    |9.0              |0.0             |149     |0.53431374|15           |1          |
|422     |-11.4666666666667|7.66666666666667|55      |0.74212521|18           |0          |
|432     |5.97166666666667 |7.76666666666667|142     |0.7723819 |20           |0          |
|528     |9.15             |13.0            |55      |0.77664191|12           |0          |
|229     |-3.025           |16.5            |85      |0.18310727|9            |0

#Feature Scaling

#Checking NaN Counts

In [8]:
feature_columns = [
    "distance",
    "temperature",
    "windspeed",
    "capacity",
    "loadfactor",
    "scheduledhour"
]
from pyspark.sql.functions import isnan, when, count, col

df.select([
    count(when(isnan(c) | col(c).isNull(), c)).alias(c)
    for c in feature_columns
]).show()

+--------+-----------+---------+--------+----------+-------------+
|distance|temperature|windspeed|capacity|loadfactor|scheduledhour|
+--------+-----------+---------+--------+----------+-------------+
|       0|         93|       93|       0|         0|            0|
+--------+-----------+---------+--------+----------+-------------+



#Removal of Rows with NaN Values

In [9]:
df_clean = df.na.drop(
    subset=[
        "distance",
        "temperature",
        "windspeed",
        "capacity",
        "loadfactor",
        "scheduledhour"
    ]
)

#Assemble and Scaling

In [10]:


feature_columns = [
    "distance",
    "temperature",
    "windspeed",
    "capacity",
    "loadfactor",
    "scheduledhour"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="assembled_features",
    handleInvalid="skip"
)

assembled_df = assembler.transform(df_clean)

scaler = StandardScaler(
    inputCol="assembled_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

scaled_df = scaler.fit(assembled_df).transform(assembled_df)

scaled_df.select(
    "assembled_features",
    "features"
).show(5, truncate=False)

+----------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------+
|assembled_features                                              |features                                                                                                                  |
+----------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------+
|[496.0,15.34,10.3,2.0,0.46060732,15.0]                          |[-0.4393484397579352,-0.30036177769833855,0.22161635203631627,-1.5902074145436875,-2.0627529425589257,0.38729734395207466]|
|[425.0,11.5,13.6666666666667,2.0,0.61790389,14.0]               |[-0.5638176290460826,-0.6891706314012466,0.8859865137371903,-1.5902074145436875,-0.8028146658888372,0.17123006094368462]  |
|[1391.0,12.2188679245283,13.5283018867925,142.0,0

#Development of Pipeline

In [14]:



# Categorical Columns

categorical_cols = [
    "origin",
    "dest",
    "uniquecarrier"
]

# String Indexers

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_index",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

# One Hot Encoders

encoders = [
    OneHotEncoder(
        inputCol=c + "_index",
        outputCol=c + "_vec"
    )
    for c in categorical_cols
]

# Numerical Features

numeric_features = [
    "distance",
    "temperature",
    "windspeed",
    "capacity",
    "loadfactor",
    "scheduledhour"
]

# Assemble Features

assembler = VectorAssembler(
    inputCols=numeric_features +
             [c + "_vec" for c in categorical_cols],
    outputCol="assembled_features",
    handleInvalid="skip"
)

# Scaling

scaler = StandardScaler(
    inputCol="assembled_features",
    outputCol="features",
    withMean=True,
    withStd=True
)


# Pipeline


pipeline = Pipeline(
    stages=indexers +
           encoders +
           [assembler, scaler]
)

# Fitting Pipeline

pipeline_model = pipeline.fit(df)

# Transformation of the Dataset

processed_df = pipeline_model.transform(df)
print("Pipeline Created Successfully")
print(pipeline.getStages())


# Showing Result

processed_df.select(
    "features",
    "delay_label"
).show(5, truncate=False)



Pipeline Created Successfully
[StringIndexer_43769ae4f33f, StringIndexer_07822c844512, StringIndexer_a2ce52bd48b2, OneHotEncoder_ea69e24b44e0, OneHotEncoder_173c166fb539, OneHotEncoder_c48488282ae9, VectorAssembler_330e8f1628ae, StandardScaler_a4c9548b681c]
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------